In [1]:
# ============================================================
# CA6000 (Kaggle PS S5E12) — Data Cleaning & Preprocessing
# Output: X_train_proc, X_val_proc, y_train, y_val, X_test_proc
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin, clone
import joblib
import scipy.sparse as sp

SEED = 42
TARGET_COL = "diagnosed_diabetes"
ID_COL = "id"


In [2]:

# ----------------------------
# 1) Robust path resolver (Kaggle / Colab / local / /mnt/data)
# ----------------------------
from pathlib import Path

def resolve_dataset_paths(prefer_dir="/content"):
    candidates = [Path(prefer_dir), Path("/mnt/data"), Path(".")]

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.append(kaggle_input)

    def find_file(root: Path, filename: str):
        direct = root / filename
        if direct.exists():
            return direct
        hits = list(root.rglob(filename))
        return hits[0] if hits else None

    train_path = test_path = sub_path = None
    for root in candidates:
        tp = find_file(root, "train.csv")
        te = find_file(root, "test.csv")
        ss = find_file(root, "sample_submission.csv")
        if tp is not None and te is not None:
            train_path, test_path, sub_path = tp, te, ss
            break

    if train_path is None or test_path is None:
        raise FileNotFoundError("Cannot find train.csv/test.csv under preferred dirs.")

    return str(train_path), str(test_path), (str(sub_path) if sub_path else None)

TRAIN_PATH, TEST_PATH, SUB_PATH = resolve_dataset_paths("/content")
print(TRAIN_PATH, TEST_PATH, SUB_PATH)

/kaggle/input/playground-series-s5e12/train.csv /kaggle/input/playground-series-s5e12/test.csv /kaggle/input/playground-series-s5e12/sample_submission.csv


In [3]:
# ----------------------------
# 2) Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\nShapes:")
print("train:", train_df.shape)
print("test :", test_df.shape)
print("\nTrain head:")
display(train_df.head(3))


Shapes:
train: (700000, 26)
test : (300000, 25)

Train head:


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0


In [4]:
# =========================
# 2.5 External ORIG features (Kaggle)
# Create: orig_mean_<col>, orig_count_<col>
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

TARGET_COL = "diagnosed_diabetes"
ID_COL = "id"

# 1) find external csv under /kaggle/input (exclude competition folder)
def find_external_csv_excluding_competition():
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError("/kaggle/input not found (not on Kaggle?)")

    all_csvs = list(root.rglob("*.csv"))
    if not all_csvs:
        raise FileNotFoundError("No csv under /kaggle/input. Did you add the dataset?")

    # exclude competition dataset
    def is_competition(p: Path) -> bool:
        s = str(p).lower()
        return "playground-series-s5e12" in s or "playground_series_s5e12" in s

    ext_csvs = [p for p in all_csvs if not is_competition(p)]
    if not ext_csvs:
        raise FileNotFoundError(
            "Found only competition csv under /kaggle/input. "
            "Please ensure you added the external dataset (Diabetes Health Indicators)."
        )

    # prefer filenames with health/indicator keyword
    prefer = [p for p in ext_csvs if ("health" in str(p).lower()) or ("indicator" in str(p).lower())]
    candidates = prefer if prefer else ext_csvs

    # choose largest file (usually main dataset)
    candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
    return str(candidates[0])

ORIG_PATH = find_external_csv_excluding_competition()
orig_df = pd.read_csv(ORIG_PATH)

print("✅ External ORIG csv:", ORIG_PATH)
print("orig_df shape:", orig_df.shape)

# 2) detect external target column
EXT_TARGET_CANDIDATES = ["Diabetes_binary", "diabetes_binary", "Diabetes_012", "diabetes_012", TARGET_COL]
ext_target = None
for c in EXT_TARGET_CANDIDATES:
    if c in orig_df.columns:
        ext_target = c
        break
if ext_target is None:
    raise ValueError(
        f"Cannot find external target column. Tried: {EXT_TARGET_CANDIDATES}. "
        f"Please check orig_df.columns."
    )

# convert external target to binary
orig_df[ext_target] = pd.to_numeric(orig_df[ext_target], errors="coerce")
if orig_df[ext_target].dropna().nunique() > 2:
    orig_df[ext_target] = (orig_df[ext_target] >= 1).astype(int)
else:
    orig_df[ext_target] = orig_df[ext_target].astype(int)

global_mean = float(orig_df[ext_target].mean())
print("✅ External target:", ext_target, "| external positive rate:", global_mean)

# 3) competition base feature columns
base_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]
common_cols = [c for c in base_cols if c in orig_df.columns]
skipped = [c for c in base_cols if c not in orig_df.columns]

print("Competition features:", len(base_cols))
print("Matched in external (exact name):", len(common_cols))
if skipped:
    print("⚠️ Skipped (not found in external by exact name):", skipped)

# 4) numeric binning helper
def quantile_edges(series: pd.Series, q=50):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    bins = pd.qcut(s, q=q, duplicates="drop")
    cats = bins.cat.categories
    if len(cats) < 2:
        return None
    edges = np.r_[cats.left[0], cats.right.values]
    edges = np.unique(edges)
    return edges if len(edges) >= 3 else None

# 5) create external features (fix fillna TypeError by casting bins to object before map)
new_cols = []
NUM_Q = 50

for col in common_cols:
    is_numeric = pd.api.types.is_numeric_dtype(orig_df[col])

    if is_numeric:
        edges = quantile_edges(orig_df[col], q=NUM_Q)
        if edges is not None:
            ext_bins = pd.cut(pd.to_numeric(orig_df[col], errors="coerce"), bins=edges, include_lowest=True)
            mean_map = orig_df.groupby(ext_bins, observed=True)[ext_target].mean()
            count_map = orig_df.groupby(ext_bins, observed=True).size()

            tr_bins = pd.cut(pd.to_numeric(train_df[col], errors="coerce"), bins=edges, include_lowest=True)
            te_bins = pd.cut(pd.to_numeric(test_df[col],  errors="coerce"), bins=edges, include_lowest=True)

            mcol = f"orig_mean_{col}"
            ccol = f"orig_count_{col}"

            train_df[mcol] = tr_bins.astype("object").map(mean_map).fillna(global_mean).astype(np.float32)
            test_df[mcol]  = te_bins.astype("object").map(mean_map).fillna(global_mean).astype(np.float32)

            train_df[ccol] = tr_bins.astype("object").map(count_map).fillna(0).astype(np.float32)
            test_df[ccol]  = te_bins.astype("object").map(count_map).fillna(0).astype(np.float32)

            new_cols += [mcol, ccol]
        else:
            # fallback for low-cardinality numeric
            mean_map = orig_df.groupby(col, observed=True)[ext_target].mean()
            count_map = orig_df[col].value_counts(dropna=False)

            mcol = f"orig_mean_{col}"
            ccol = f"orig_count_{col}"

            train_df[mcol] = train_df[col].map(mean_map).fillna(global_mean).astype(np.float32)
            test_df[mcol]  = test_df[col].map(mean_map).fillna(global_mean).astype(np.float32)

            train_df[ccol] = train_df[col].map(count_map).fillna(0).astype(np.float32)
            test_df[ccol]  = test_df[col].map(count_map).fillna(0).astype(np.float32)

            new_cols += [mcol, ccol]
    else:
        mean_map = orig_df.groupby(col, observed=True)[ext_target].mean()
        count_map = orig_df[col].value_counts(dropna=False)

        mcol = f"orig_mean_{col}"
        ccol = f"orig_count_{col}"

        train_df[mcol] = train_df[col].map(mean_map).fillna(global_mean).astype(np.float32)
        test_df[mcol]  = test_df[col].map(mean_map).fillna(global_mean).astype(np.float32)

        train_df[ccol] = train_df[col].map(count_map).fillna(0).astype(np.float32)
        test_df[ccol]  = test_df[col].map(count_map).fillna(0).astype(np.float32)

        new_cols += [mcol, ccol]

# log1p on external count features to reduce scale/overfit
count_cols = [c for c in train_df.columns if c.startswith("orig_count_")]
if count_cols:
    train_df[count_cols] = np.log1p(train_df[count_cols].astype(np.float32))
    test_df[count_cols]  = np.log1p(test_df[count_cols].astype(np.float32))
print("log1p on count cols:", len(count_cols))

print(f"✅ Added {len(new_cols)} external features.")
print("train_df shape now:", train_df.shape)
print("test_df  shape now:", test_df.shape)
print("Example new cols:", new_cols[:10])

print("ORIG_PATH =", ORIG_PATH)
print("orig_df.shape =", orig_df.shape)


✅ External ORIG csv: /kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
orig_df shape: (100000, 31)
✅ External target: diagnosed_diabetes | external positive rate: 0.59998
Competition features: 24
Matched in external (exact name): 24
log1p on count cols: 24
✅ Added 48 external features.
train_df shape now: (700000, 74)
test_df  shape now: (300000, 73)
Example new cols: ['orig_mean_age', 'orig_count_age', 'orig_mean_alcohol_consumption_per_week', 'orig_count_alcohol_consumption_per_week', 'orig_mean_physical_activity_minutes_per_week', 'orig_count_physical_activity_minutes_per_week', 'orig_mean_diet_score', 'orig_count_diet_score', 'orig_mean_sleep_hours_per_day', 'orig_count_sleep_hours_per_day']
ORIG_PATH = /kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
orig_df.shape = (100000, 31)


In [5]:
# ----------------------------
# 3) Data audit & sanity checks (good for your report)
# ----------------------------
def basic_audit(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # Required columns
    assert TARGET_COL in train_df.columns, f"Missing target '{TARGET_COL}' in train.csv"
    assert ID_COL in train_df.columns and ID_COL in test_df.columns, "Missing 'id' in train/test"

    # Column alignment (except target)
    train_features = [c for c in train_df.columns if c != TARGET_COL]
    assert set(train_features) == set(test_df.columns), "Train features != Test columns (schema mismatch)"

    # ID uniqueness
    assert train_df[ID_COL].is_unique, "Train id is not unique"
    assert test_df[ID_COL].is_unique, "Test id is not unique"

    # Duplicates
    dup_train = train_df.duplicated().sum()
    dup_test = test_df.duplicated().sum()

    # Missing summary
    miss_train = train_df.isnull().mean().sort_values(ascending=False)
    miss_test = test_df.isnull().mean().sort_values(ascending=False)

    # Target check
    y = train_df[TARGET_COL]
    unique_y = sorted(y.dropna().unique().tolist())

    print("\n[Audit] duplicates:", {"train": int(dup_train), "test": int(dup_test)})
    print("[Audit] top missing rate (train):")
    print(miss_train.head(10))
    print("[Audit] top missing rate (test):")
    print(miss_test.head(10))
    print("[Audit] target unique values:", unique_y)
    print("[Audit] target distribution:\n", y.value_counts(dropna=False))

    # ✅ NEW: external feature sanity check
    ext_cols = [c for c in train_df.columns if c.startswith("orig_mean_") or c.startswith("orig_count_")]
    print("\n[Audit] external feature count:", len(ext_cols))
    if len(ext_cols) == 0:
        print("⚠️ No external features detected.")
        print("   -> Make sure Cell 2.5 ran BEFORE this audit cell, and it added orig_* columns to BOTH train_df and test_df.")
    else:
        # ensure external cols exist in both train/test
        missing_in_test = [c for c in ext_cols if c not in test_df.columns]
        if missing_in_test:
            raise AssertionError(f"External cols missing in test_df (schema break): {missing_in_test[:10]}")

        # show quick stats (prove they are populated and numeric)
        print("[Audit] external missing rate (train avg):", float(train_df[ext_cols].isna().mean().mean()))
        print("[Audit] external missing rate (test  avg):", float(test_df[ext_cols].isna().mean().mean()))
        print("[Audit] external feature sample:", ext_cols[:6])
        print("[Audit] external feature stats (train, first 6):")
        print(train_df[ext_cols[:6]].describe().T[["mean", "std", "min", "max"]])

basic_audit(train_df, test_df)

# Convert target to int (0/1)
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)


[Audit] duplicates: {'train': 0, 'test': 0}
[Audit] top missing rate (train):
id                                    0.0
age                                   0.0
alcohol_consumption_per_week          0.0
physical_activity_minutes_per_week    0.0
diet_score                            0.0
sleep_hours_per_day                   0.0
screen_time_hours_per_day             0.0
bmi                                   0.0
waist_to_hip_ratio                    0.0
systolic_bp                           0.0
dtype: float64
[Audit] top missing rate (test):
id                                    0.0
age                                   0.0
alcohol_consumption_per_week          0.0
physical_activity_minutes_per_week    0.0
diet_score                            0.0
sleep_hours_per_day                   0.0
screen_time_hours_per_day             0.0
bmi                                   0.0
waist_to_hip_ratio                    0.0
systolic_bp                           0.0
dtype: float64
[Audit] target uni

In [6]:

# ----------------------------
# 4) Define column groups
# ----------------------------
# Categorical columns (object/string)
cat_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != ID_COL]  # ensure id is not treated as category

# Binary columns (known 0/1 flags in this dataset)
bin_cols = ["family_history_diabetes", "hypertension_history", "cardiovascular_history"]
bin_cols = [c for c in bin_cols if c in train_df.columns]

# Numeric columns = all numeric excluding id/target/binary
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in [ID_COL, TARGET_COL] + bin_cols]

print("Column groups:")
print("num_cols:", num_cols)
print("bin_cols:", bin_cols)
print("cat_cols:", cat_cols)


Column groups:
num_cols: ['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'orig_mean_age', 'orig_count_age', 'orig_mean_alcohol_consumption_per_week', 'orig_count_alcohol_consumption_per_week', 'orig_mean_physical_activity_minutes_per_week', 'orig_count_physical_activity_minutes_per_week', 'orig_mean_diet_score', 'orig_count_diet_score', 'orig_mean_sleep_hours_per_day', 'orig_count_sleep_hours_per_day', 'orig_mean_screen_time_hours_per_day', 'orig_count_screen_time_hours_per_day', 'orig_mean_bmi', 'orig_count_bmi', 'orig_mean_waist_to_hip_ratio', 'orig_count_waist_to_hip_ratio', 'orig_mean_systolic_bp', 'orig_count_systolic_bp', 'orig_mean_diastolic_bp', 'orig_count_diastolic_bp', 'orig_mean_heart_rate', 'orig_count_heart_rate', 'orig_mean_chole

In [7]:
# Optional: verify binary columns truly contain only 0/1
for c in bin_cols:
    bad_vals = set(train_df[c].dropna().unique()) - {0, 1}
    if bad_vals:
        raise ValueError(f"Binary col '{c}' has unexpected values: {bad_vals}")

In [8]:
# ----------------------------
# 5) (Optional but nice) Range check for numeric columns
# ----------------------------
def numeric_range_report(df: pd.DataFrame, columns):
    desc = df[columns].describe(percentiles=[0.01, 0.5, 0.99]).T
    # Keep a compact view
    return desc[["min", "1%", "50%", "99%", "max", "mean", "std"]].sort_values("max", ascending=False)

range_report = numeric_range_report(train_df, num_cols)
print("\nNumeric range report (top 8 by max):")
display(range_report.head(8))



Numeric range report (top 8 by max):


,min,1%,50%,99%,max,mean,std
physical_activity_minutes_per_week,1.0,16.0,71.0,304.0,747.0,80.230803,51.195071
triglycerides,31.0,67.0,123.0,187.0,290.0,123.081850,24.739397
cholesterol_total,117.0,150.0,187.0,225.0,289.0,186.818801,16.730832
ldl_cholesterol,51.0,61.0,103.0,148.0,205.0,102.905854,19.022416
systolic_bp,91.0,93.0,116.0,141.0,163.0,116.294193,11.010390
diastolic_bp,51.0,60.0,75.0,91.0,104.0,75.440924,6.825775
heart_rate,42.0,54.0,70.0,86.0,101.0,70.167749,6.938722
hdl_cholesterol,21.0,35.0,54.0,73.0,90.0,53.823214,8.266545


In [9]:
# ----------------------------
# 6) Split data BEFORE fitting preprocessors (avoid leakage)
# ----------------------------
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].values.astype(np.int32)

train_ids = X[ID_COL].values
test_ids = test_df[ID_COL].values

X = X.drop(columns=[ID_COL])
X_test = test_df.drop(columns=[ID_COL])

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("\nSplit shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape)



Split shapes:
X_train: (560000, 72) y_train: (560000,)
X_val  : (140000, 72) y_val  : (140000,)
X_test : (300000, 72)


In [10]:
# ----------------------------
# 7) Custom transformer: quantile clipping for numeric outliers
#    (fit on training only)
# ----------------------------
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q=0.005, upper_q=0.995):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(X, self.upper_q, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)

In [11]:
# ----------------------------
# 8) Build preprocessing pipeline
#    - numeric: median impute -> clip -> standardize
#    - binary : most_frequent impute (keep 0/1)
#    - cate   : most_frequent impute -> Fold-safe target encoding (mean + count)
# ----------------------------

class MeanCountTargetEncoder(BaseEstimator, TransformerMixin):
    """Leakage-safe when fitted ONLY on training fold (we will do that in CV loop).
    Produces 2 features per categorical column: smoothed mean target + frequency(count).
    """
    def __init__(self, smoothing: float = 20.0):
        self.smoothing = smoothing

    def fit(self, X, y):
        X = pd.DataFrame(X).astype("object")
        y = pd.Series(y)

        self.global_mean_ = float(y.mean())
        self.mean_maps_ = []
        self.count_maps_ = []

        for j in range(X.shape[1]):
            col = X.iloc[:, j]
            stats = y.groupby(col).agg(["mean", "count"])
            smooth_mean = (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing) / (stats["count"] + self.smoothing)

            self.mean_maps_.append(smooth_mean)
            self.count_maps_.append(stats["count"])

        return self

    def transform(self, X):
        X = pd.DataFrame(X).astype("object")

        out_cols = []
        for j in range(X.shape[1]):
            col = X.iloc[:, j]
            m = col.map(self.mean_maps_[j]).fillna(self.global_mean_).astype(np.float32).to_numpy()
            c = col.map(self.count_maps_[j]).fillna(0).astype(np.float32).to_numpy()
            out_cols.append(m)
            out_cols.append(c)

        return np.vstack(out_cols).T  # (n_samples, 2 * n_cat_cols)


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", QuantileClipper(lower_q=0.005, upper_q=0.995)),
    ("scaler", StandardScaler())
])

binary_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("te", MeanCountTargetEncoder(smoothing=20.0))
])

# NOTE:
# - After switching OneHot -> TargetEncoding, output becomes dense (numpy array).
# - This is OK for XGBoost and also convenient for neural nets.
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("bin", binary_pipe, bin_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.0,  # force dense output
    verbose_feature_names_out=False
)

In [12]:
# ----------------------------
# 9) Fit on train, transform val/test
# IMPORTANT: pass y_train into fit_transform because categorical target encoder needs y
# ----------------------------
X_train_proc = preprocess.fit_transform(X_train, y_train)
X_val_proc = preprocess.transform(X_val)
X_test_proc = preprocess.transform(X_test)

# Cast to float32 for models (works for dense or sparse matrices)
X_train_proc = X_train_proc.astype(np.float32)
X_val_proc = X_val_proc.astype(np.float32)
X_test_proc = X_test_proc.astype(np.float32)

print("Processed shapes:")
print("X_train_proc:", X_train_proc.shape)
print("X_val_proc  :", X_val_proc.shape)
print("X_test_proc :", X_test_proc.shape)

# Safety checks (support sparse/dense)
def assert_no_nan(arr, name):
    if sp.issparse(arr):
        assert not np.isnan(arr.data).any(), f"NaNs remain in {name}"
    else:
        assert not np.isnan(arr).any(), f"NaNs remain in {name}"

assert_no_nan(X_train_proc, "X_train_proc")
assert_no_nan(X_val_proc, "X_val_proc")
assert_no_nan(X_test_proc, "X_test_proc")

Processed shapes:
X_train_proc: (560000, 78)
X_val_proc  : (140000, 78)
X_test_proc : (300000, 78)


In [13]:
# ----------------------------
# 10) Save artifacts for reproducibility
# ----------------------------
artifact = {
    "id_col": ID_COL,
    "target_col": TARGET_COL,
    "num_cols": num_cols,
    "bin_cols": bin_cols,
    "cat_cols": cat_cols,
    "preprocess": preprocess,
}

joblib.dump(artifact, "preprocess_artifact.joblib")
print("\nSaved preprocess artifact -> preprocess_artifact.joblib")

# Optional: save processed arrays (may be large, enable if you want)
# np.save("X_train_proc.npy", X_train_proc)
# np.save("X_val_proc.npy", X_val_proc)
# np.save("X_test_proc.npy", X_test_proc)
# np.save("y_train.npy", y_train)
# np.save("y_val.npy", y_val)

print("\n✅ Ready for model training stage:")
print("Use X_train_proc, y_train, X_val_proc, y_val, X_test_proc")


Saved preprocess artifact -> preprocess_artifact.joblib

✅ Ready for model training stage:
Use X_train_proc, y_train, X_val_proc, y_val, X_test_proc


In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

SEED = 42
N_SPLITS = 5  # 5-fold

y_full = train_df[TARGET_COL].astype(int).values
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = list(skf.split(np.zeros(len(y_full)), y_full))

print("folds:", len(folds), "example sizes:", [(len(tr), len(va)) for tr, va in folds[:2]])


In [ ]:
# ----------------------------
# 11) 5-fold OOF XGBoost (AUC) + Test bagging
# NOTE: preprocess contains target encoder, so we MUST call fit_transform(X_tr, y_tr) inside each fold.
# ----------------------------
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.base import clone


def xgb_cuda_ok():
    try:
        X_tmp = np.random.randn(200, 8).astype(np.float32)
        y_tmp = np.random.randint(0, 2, 200).astype(np.int32)
        m = xgb.XGBClassifier(
            n_estimators=10,
            tree_method="hist",
            device="cuda",
            objective="binary:logistic",
            eval_metric="auc",
        )
        m.fit(X_tmp, y_tmp)
        return True
    except Exception as e:
        print("[XGB] CUDA test failed -> CPU fallback:", str(e)[:200])
        return False


USE_XGB_CUDA = xgb_cuda_ok()

X_full_raw = train_df.drop(columns=[TARGET_COL, ID_COL])
X_test_raw = test_df.drop(columns=[ID_COL])

oof_xgb = np.zeros(len(X_full_raw), dtype=np.float32)
test_xgb = np.zeros(len(X_test_raw), dtype=np.float32)

xgb_params = dict(
    n_estimators=50000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=SEED,
    early_stopping_rounds=200,
)
if USE_XGB_CUDA:
    xgb_params["device"] = "cuda"

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr_raw, X_va_raw = X_full_raw.iloc[tr_idx], X_full_raw.iloc[va_idx]
    y_tr, y_va = y_full[tr_idx], y_full[va_idx]

    pre = clone(preprocess)
    X_tr = pre.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va = pre.transform(X_va_raw).astype(np.float32)
    X_te = pre.transform(X_test_raw).astype(np.float32)

    pos = int((y_tr == 1).sum())
    neg = int((y_tr == 0).sum())
    scale_pos = neg / max(pos, 1)

    model = xgb.XGBClassifier(**xgb_params, scale_pos_weight=scale_pos)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=200)

    best_iter = getattr(model, "best_iteration", None)
    if best_iter is not None:
        p_va = model.predict_proba(X_va, iteration_range=(0, best_iter + 1))[:, 1]
        p_te = model.predict_proba(X_te, iteration_range=(0, best_iter + 1))[:, 1]
    else:
        p_va = model.predict_proba(X_va)[:, 1]
        p_te = model.predict_proba(X_te)[:, 1]

    oof_xgb[va_idx] = p_va.astype(np.float32)
    test_xgb += p_te.astype(np.float32) / N_SPLITS

    print(f"[XGB] fold {fold} AUC =", roc_auc_score(y_va, p_va))

print("[XGB] OOF AUC =", roc_auc_score(y_full, oof_xgb))


In [ ]:
# CatBoost-native input (keep categorical columns)
X_cb = train_df.drop(columns=[TARGET_COL, ID_COL]).copy()
y_cb = train_df[TARGET_COL].astype(int).values
X_cb_test = test_df.drop(columns=[ID_COL]).copy()

cat_features = [c for c in cat_cols if c in X_cb.columns]

for c in cat_features:
    X_cb[c] = X_cb[c].astype("string").fillna("missing")
    X_cb_test[c] = X_cb_test[c].astype("string").fillna("missing")

print("CatBoost input:", X_cb.shape, X_cb_test.shape, "cat_features:", len(cat_features), "pos rate =", y_cb.mean())


In [16]:
!pip -q install catboost

import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

oof_cat = np.zeros(len(X_cb), dtype=np.float32)
test_cat = np.zeros(len(X_cb_test), dtype=np.float32)

cat_params_gpu = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=20000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=10,
    auto_class_weights="Balanced",

    task_type="GPU",
    devices="0",
    boosting_type="Plain",
    bootstrap_type="Bayesian",
    bagging_temperature=0.7,
    random_strength=1.0,
    # rsm=0.8,
    min_data_in_leaf=64,
    border_count=254,

    max_ctr_complexity=4,
    # ctr_leaf_count_limit=128,
    verbose=200,
    random_seed=SEED,
)

# Optional CPU Ordered version (slower but often more stable on cats)
# cat_params_cpu = dict(
#     loss_function="Logloss",
#     eval_metric="AUC",
#     iterations=20000,
#     learning_rate=0.03,
#     depth=8,
#     l2_leaf_reg=10,
#     auto_class_weights="Balanced",
#     boosting_type="Ordered",
#     bootstrap_type="Bayesian",
#     bagging_temperature=0.7,
#     random_strength=1.0,
#     rsm=0.8,
#     min_data_in_leaf=64,
#     border_count=254,
#     max_ctr_complexity=4,
#     ctr_leaf_count_limit=128,
#     verbose=200,
#     random_seed=SEED,
# )

cat_params = cat_params_gpu

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr, X_va = X_cb.iloc[tr_idx], X_cb.iloc[va_idx]
    y_tr, y_va = y_cb[tr_idx], y_cb[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_cb_test, cat_features=cat_features)

    model = CatBoostClassifier(**cat_params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=300,
    )

    oof_cat[va_idx] = model.predict_proba(valid_pool)[:, 1].astype(np.float32)
    test_cat += model.predict_proba(test_pool)[:, 1].astype(np.float32) / N_SPLITS

    auc = roc_auc_score(y_va, oof_cat[va_idx])
    print(f"[CatBoost-native] fold {fold} AUC = {auc:.5f}")

print("[CatBoost-native] OOF AUC =", roc_auc_score(y_cb, oof_cat))


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6907147	best: 0.6907147 (0)	total: 13.1s	remaining: 3d 49m 38s
200:	test: 0.7135000	best: 0.7135000 (200)	total: 26.4s	remaining: 43m 16s
400:	test: 0.7182831	best: 0.7182831 (400)	total: 39.6s	remaining: 32m 16s
600:	test: 0.7218610	best: 0.7218610 (600)	total: 52.8s	remaining: 28m 25s
800:	test: 0.7238004	best: 0.7238044 (799)	total: 1m 6s	remaining: 26m 25s
1000:	test: 0.7246842	best: 0.7246873 (998)	total: 1m 19s	remaining: 25m 7s
1200:	test: 0.7253402	best: 0.7253509 (1197)	total: 1m 32s	remaining: 24m 9s
1400:	test: 0.7256593	best: 0.7256593 (1400)	total: 1m 45s	remaining: 23m 23s
1600:	test: 0.7259642	best: 0.7259673 (1599)	total: 1m 58s	remaining: 22m 47s
1800:	test: 0.7262013	best: 0.7262079 (1787)	total: 2m 12s	remaining: 22m 16s
2000:	test: 0.7264704	best: 0.7264704 (2000)	total: 2m 25s	remaining: 21m 48s
2200:	test: 0.7265581	best: 0.7265594 (2189)	total: 2m 38s	remaining: 21m 23s
2400:	test: 0.7266973	best: 0.7267063 (2399)	total: 2m 51s	remaining: 21m
2600:	tes

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6886771	best: 0.6886771 (0)	total: 71ms	remaining: 23m 38s
200:	test: 0.7110051	best: 0.7110051 (200)	total: 13.3s	remaining: 21m 52s
400:	test: 0.7159503	best: 0.7159503 (400)	total: 26.6s	remaining: 21m 39s
600:	test: 0.7196409	best: 0.7196409 (600)	total: 39.7s	remaining: 21m 22s
800:	test: 0.7214230	best: 0.7214230 (800)	total: 53s	remaining: 21m 9s
1000:	test: 0.7225455	best: 0.7225455 (1000)	total: 1m 6s	remaining: 20m 55s
1200:	test: 0.7232293	best: 0.7232358 (1193)	total: 1m 19s	remaining: 20m 41s
1400:	test: 0.7237862	best: 0.7237862 (1400)	total: 1m 32s	remaining: 20m 28s
1600:	test: 0.7240695	best: 0.7240707 (1599)	total: 1m 45s	remaining: 20m 15s
1800:	test: 0.7243249	best: 0.7243254 (1799)	total: 1m 58s	remaining: 20m 2s
2000:	test: 0.7245152	best: 0.7245173 (1998)	total: 2m 12s	remaining: 19m 49s
2200:	test: 0.7246504	best: 0.7246504 (2200)	total: 2m 25s	remaining: 19m 36s
2400:	test: 0.7248113	best: 0.7248123 (2399)	total: 2m 38s	remaining: 19m 22s
2600:	test:

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6891555	best: 0.6891555 (0)	total: 71ms	remaining: 23m 39s
200:	test: 0.7112945	best: 0.7112945 (200)	total: 13.4s	remaining: 21m 55s
400:	test: 0.7165850	best: 0.7165850 (400)	total: 26.6s	remaining: 21m 39s
600:	test: 0.7205482	best: 0.7205482 (600)	total: 39.8s	remaining: 21m 26s
800:	test: 0.7224672	best: 0.7224672 (800)	total: 53.1s	remaining: 21m 12s
1000:	test: 0.7235710	best: 0.7235717 (995)	total: 1m 6s	remaining: 20m 58s
1200:	test: 0.7242718	best: 0.7242718 (1200)	total: 1m 19s	remaining: 20m 43s
1400:	test: 0.7247603	best: 0.7247629 (1391)	total: 1m 32s	remaining: 20m 30s
1600:	test: 0.7251530	best: 0.7251530 (1600)	total: 1m 46s	remaining: 20m 18s
1800:	test: 0.7254340	best: 0.7254396 (1797)	total: 1m 59s	remaining: 20m 5s
2000:	test: 0.7256735	best: 0.7256739 (1993)	total: 2m 12s	remaining: 19m 52s
2200:	test: 0.7258931	best: 0.7258931 (2200)	total: 2m 25s	remaining: 19m 38s
2400:	test: 0.7260144	best: 0.7260253 (2392)	total: 2m 38s	remaining: 19m 24s
2600:	tes

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6900977	best: 0.6900977 (0)	total: 70.4ms	remaining: 23m 27s
200:	test: 0.7129781	best: 0.7129781 (200)	total: 13.3s	remaining: 21m 46s
400:	test: 0.7174721	best: 0.7174721 (400)	total: 26.4s	remaining: 21m 31s
600:	test: 0.7215172	best: 0.7215172 (600)	total: 39.6s	remaining: 21m 17s
800:	test: 0.7233576	best: 0.7233576 (800)	total: 52.8s	remaining: 21m 4s
1000:	test: 0.7243354	best: 0.7243369 (998)	total: 1m 5s	remaining: 20m 49s
1200:	test: 0.7251107	best: 0.7251107 (1200)	total: 1m 19s	remaining: 20m 37s
1400:	test: 0.7254964	best: 0.7255013 (1388)	total: 1m 32s	remaining: 20m 24s
1600:	test: 0.7257327	best: 0.7257405 (1591)	total: 1m 45s	remaining: 20m 12s
1800:	test: 0.7261149	best: 0.7261354 (1791)	total: 1m 58s	remaining: 19m 59s
2000:	test: 0.7262360	best: 0.7262467 (1980)	total: 2m 11s	remaining: 19m 47s
2200:	test: 0.7263300	best: 0.7263518 (2152)	total: 2m 25s	remaining: 19m 34s
2400:	test: 0.7263497	best: 0.7263576 (2374)	total: 2m 38s	remaining: 19m 21s
2600:	t

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6895891	best: 0.6895891 (0)	total: 69.3ms	remaining: 23m 6s
200:	test: 0.7124377	best: 0.7124377 (200)	total: 13.3s	remaining: 21m 47s
400:	test: 0.7174238	best: 0.7174238 (400)	total: 26.6s	remaining: 21m 39s
600:	test: 0.7209939	best: 0.7209940 (599)	total: 39.8s	remaining: 21m 24s
800:	test: 0.7229121	best: 0.7229121 (800)	total: 53s	remaining: 21m 10s
1000:	test: 0.7239614	best: 0.7239614 (1000)	total: 1m 6s	remaining: 20m 56s
1200:	test: 0.7247251	best: 0.7247251 (1200)	total: 1m 19s	remaining: 20m 46s
1400:	test: 0.7253066	best: 0.7253066 (1400)	total: 1m 32s	remaining: 20m 33s
1600:	test: 0.7257353	best: 0.7257379 (1599)	total: 1m 46s	remaining: 20m 20s
1800:	test: 0.7260602	best: 0.7260661 (1794)	total: 1m 59s	remaining: 20m 7s
2000:	test: 0.7261859	best: 0.7262077 (1978)	total: 2m 12s	remaining: 19m 54s
2200:	test: 0.7263411	best: 0.7263509 (2196)	total: 2m 26s	remaining: 19m 42s
2400:	test: 0.7264370	best: 0.7264370 (2400)	total: 2m 39s	remaining: 19m 29s
2600:	tes

In [ ]:
!pip -q install lightgbm
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np

X_lgb = train_df.drop(columns=[TARGET_COL, ID_COL]).copy()
y_lgb = train_df[TARGET_COL].astype(int).values
X_lgb_test = test_df.drop(columns=[ID_COL]).copy()

cat_features = [c for c in cat_cols if c in X_lgb.columns]
for c in cat_features:
    X_lgb[c] = X_lgb[c].astype("category")
    X_lgb_test[c] = X_lgb_test[c].astype("category")

oof_lgb = np.zeros(len(X_lgb), dtype=np.float32)
test_lgb = np.zeros(len(X_lgb_test), dtype=np.float32)

lgb_params = dict(
    n_estimators=50000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    min_child_samples=200,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary",
    metric="auc",
    n_jobs=-1,
    random_state=SEED,
)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr, X_va = X_lgb.iloc[tr_idx], X_lgb.iloc[va_idx]
    y_tr, y_va = y_lgb[tr_idx], y_lgb[va_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(200)],
    )

    p_va = model.predict_proba(X_va, num_iteration=model.best_iteration_)[:, 1]
    p_te = model.predict_proba(X_lgb_test, num_iteration=model.best_iteration_)[:, 1]

    oof_lgb[va_idx] = p_va.astype(np.float32)
    test_lgb += p_te.astype(np.float32) / N_SPLITS

    print(f"[LGB] fold {fold} AUC =", roc_auc_score(y_va, p_va))

print("[LGB] OOF AUC =", roc_auc_score(y_lgb, oof_lgb))


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)


class TabMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp_one_fold(X_tr, y_tr, X_va, y_va, X_te, epochs=10, bs=8192, lr=1e-3, patience=2):
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    X_va_t = torch.tensor(X_va, dtype=torch.float32)
    y_va_t = torch.tensor(y_va, dtype=torch.float32)
    X_te_t = torch.tensor(X_te, dtype=torch.float32)

    tr_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=bs, shuffle=True, num_workers=2)
    va_loader = DataLoader(TensorDataset(X_va_t, y_va_t), batch_size=bs, shuffle=False, num_workers=2)
    te_loader = DataLoader(TensorDataset(X_te_t), batch_size=bs, shuffle=False, num_workers=2)

    model = TabMLP(X_tr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        va_pred = []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device)
                p = torch.sigmoid(model(xb)).detach().cpu().numpy()
                va_pred.append(p)
        va_pred = np.concatenate(va_pred)
        auc = roc_auc_score(y_va, va_pred)
        print(f"[MLP] ep {ep} val AUC = {auc:.5f}")

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    va_pred = []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb = xb.to(device)
            va_pred.append(torch.sigmoid(model(xb)).detach().cpu().numpy())
    va_pred = np.concatenate(va_pred)

    te_pred = []
    with torch.no_grad():
        for (xb,) in te_loader:
            xb = xb.to(device)
            te_pred.append(torch.sigmoid(model(xb)).detach().cpu().numpy())
    te_pred = np.concatenate(te_pred)

    return va_pred.astype(np.float32), te_pred.astype(np.float32)


X_full_raw = train_df.drop(columns=[TARGET_COL, ID_COL])
X_test_raw = test_df.drop(columns=[ID_COL])

oof_mlp = np.zeros(len(X_full_raw), dtype=np.float32)
test_mlp = np.zeros(len(X_test_raw), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr_raw, X_va_raw = X_full_raw.iloc[tr_idx], X_full_raw.iloc[va_idx]
    y_tr, y_va = y_full[tr_idx], y_full[va_idx]

    pre = clone(preprocess)
    X_tr = pre.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va = pre.transform(X_va_raw).astype(np.float32)
    X_te = pre.transform(X_test_raw).astype(np.float32)

    p_va, p_te = train_mlp_one_fold(X_tr, y_tr, X_va, y_va, X_te)
    oof_mlp[va_idx] = p_va
    test_mlp += p_te / N_SPLITS

    print(f"[MLP] fold {fold} AUC =", roc_auc_score(y_va, p_va))

print("[MLP] OOF AUC =", roc_auc_score(y_full, oof_mlp))


In [17]:
# FT-Transformer code disabled to focus on GPU CatBoost.
# Uncomment the block below if you want to run FT again.
#
# import numpy as np
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# from sklearn.metrics import roc_auc_score
# from sklearn.base import clone
#
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("device:", device)
#
# def assert_finite(name, X):
#     if not np.isfinite(X).all():
#         bad = np.where(~np.isfinite(X))
#         print(name, "non-finite count:", len(bad[0]))
#         raise ValueError(f"{name} contains NaN/Inf")
#
# class NumpyDataset(Dataset):
#     def __init__(self, X, y=None):
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = None if y is None else torch.tensor(y, dtype=torch.float32)
#     def __len__(self): return self.X.shape[0]
#     def __getitem__(self, i):
#         if self.y is None: return self.X[i]
#         return self.X[i], self.y[i]
#
# class FTTransformer(nn.Module):
#     def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
#         super().__init__()
#         self.W = nn.Parameter(torch.randn(n_features, d_token) * 0.01)
#         self.b = nn.Parameter(torch.zeros(n_features, d_token))
#         self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
#
#         enc = nn.TransformerEncoderLayer(
#             d_model=d_token, nhead=n_heads, dim_feedforward=d_token*4,
#             dropout=dropout, batch_first=True, activation="gelu"
#         )
#         self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
#
#         self.head = nn.Sequential(
#             nn.LayerNorm(d_token),
#             nn.Linear(d_token, d_token),
#             nn.GELU(),
#             nn.Dropout(dropout),
#             nn.Linear(d_token, 1)
#         )
#
#     def forward(self, x):
#         tokens = x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)
#         cls = self.cls.expand(x.size(0), -1, -1)
#         tokens = torch.cat([cls, tokens], dim=1)
#         z = self.encoder(tokens)
#         logits = self.head(z[:, 0, :]).squeeze(-1)
#         return logits
#
# def train_ft_once(X_tr, y_tr, X_va, y_va, X_te, epochs=5, batch_size=4096, lr=5e-4):
#     tr_loader = DataLoader(NumpyDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, num_workers=2)
#     va_loader = DataLoader(NumpyDataset(X_va, y_va), batch_size=batch_size, shuffle=False, num_workers=2)
#     te_loader = DataLoader(NumpyDataset(X_te, None), batch_size=batch_size, shuffle=False, num_workers=2)
#
#     model = FTTransformer(n_features=X_tr.shape[1], d_token=64, n_heads=8, n_layers=3, dropout=0.1).to(device)
#     opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
#     loss_fn = nn.BCEWithLogitsLoss()
#
#     best_auc, best_state = -1, None
#
#     for ep in range(1, epochs + 1):
#         model.train()
#         for xb, yb in tr_loader:
#             xb, yb = xb.to(device), yb.to(device)
#             opt.zero_grad()
#             logits = model(xb)
#             loss = loss_fn(logits, yb)
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # ⭐
#             opt.step()
#
#         model.eval()
#         va_pred = []
#         with torch.no_grad():
#             for xb, yb in va_loader:
#                 xb = xb.to(device)
#                 p = torch.sigmoid(model(xb)).cpu().numpy()
#                 va_pred.append(p)
#         va_pred = np.concatenate(va_pred)
#
#         if not np.isfinite(va_pred).all():
#             raise ValueError("va_pred contains NaN/Inf (training unstable). Try smaller lr or fewer layers.")
#
#         auc = roc_auc_score(y_va, va_pred)
#         print(f"[FT] epoch {ep} val AUC = {auc:.5f}")
#
#         if auc > best_auc:
#             best_auc = auc
#             best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#
#     model.load_state_dict(best_state)
#     model.to(device)
#     model.eval()
#
#     # final val pred
#     va_pred = []
#     with torch.no_grad():
#         for xb, yb in va_loader:
#             xb = xb.to(device)
#             va_pred.append(torch.sigmoid(model(xb)).cpu().numpy())
#     va_pred = np.concatenate(va_pred)
#
#     # test pred
#     te_pred = []
#     with torch.no_grad():
#         for xb in te_loader:
#             xb = xb.to(device)
#             te_pred.append(torch.sigmoid(model(xb)).cpu().numpy())
#     te_pred = np.concatenate(te_pred)
#
#     return va_pred.astype(np.float32), te_pred.astype(np.float32), best_auc
#
# # 5-fold
# oof_ft = np.zeros(len(X_full_raw), dtype=np.float32)
# test_ft = np.zeros(len(X_test_raw), dtype=np.float32)
#
# for fold, (tr_idx, va_idx) in enumerate(folds, 1):
#     X_tr_raw = X_full_raw.iloc[tr_idx]
#     X_va_raw = X_full_raw.iloc[va_idx]
#     y_tr = y_full[tr_idx]
#     y_va = y_full[va_idx]
#
#     pre = clone(preprocess)
#     X_tr = pre.fit_transform(X_tr_raw, y_tr).astype(np.float32)
#     X_va = pre.transform(X_va_raw).astype(np.float32)
#     X_te = pre.transform(X_test_raw).astype(np.float32)
#
#     assert_finite("X_tr", X_tr)
#     assert_finite("X_va", X_va)
#     assert_finite("X_te", X_te)
#
#     va_pred, te_pred, best_auc = train_ft_once(
#         X_tr, y_tr, X_va, y_va, X_te,
#         epochs=5, batch_size=4096, lr=5e-4
#     )
#
#     oof_ft[va_idx] = va_pred
#     test_ft += te_pred / N_SPLITS
#     print(f"[FT] fold {fold} best AUC = {best_auc:.5f}")
#
# print("[FT] OOF AUC =", roc_auc_score(y_full, oof_ft))


In [ ]:
import numpy as np


def check_oof(name, oof):
    bad = np.where(~np.isfinite(oof))[0]
    print(name, "non-finite:", len(bad))
    assert len(bad) == 0, f"{name} has NaN/Inf (likely some fold didn't write va_idx)"


check_oof("oof_xgb", oof_xgb)
check_oof("oof_cat", oof_cat)
check_oof("oof_lgb", oof_lgb)
check_oof("oof_mlp", oof_mlp)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

assert len(oof_xgb) == len(oof_cat) == len(oof_lgb) == len(oof_mlp) == len(y_full)

S_train = np.column_stack([oof_xgb, oof_cat, oof_lgb, oof_mlp]).astype(np.float32)
S_test = np.column_stack([test_xgb, test_cat, test_lgb, test_mlp]).astype(np.float32)

meta = LogisticRegression(max_iter=5000, random_state=SEED)
meta.fit(S_train, y_full)

stack_oof = meta.predict_proba(S_train)[:, 1]
print("[STACK] OOF AUC =", roc_auc_score(y_full, stack_oof))

stack_test = meta.predict_proba(S_test)[:, 1].astype(np.float32)

submission = pd.DataFrame({ID_COL: test_df[ID_COL].values, TARGET_COL: stack_test})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv", submission.shape)


In [ ]:
import os
print("cwd:", os.getcwd())
print("exists submission.csv:", os.path.exists("submission.csv"))
!ls -lh


In [19]:
print(submission.isna().sum())
print(submission[TARGET_COL].min(), submission[TARGET_COL].max())
print(submission[ID_COL].nunique(), submission.shape[0])

id                    0
diagnosed_diabetes    0
dtype: int64
0.1483093649148941 0.9439290165901184
300000 300000


In [20]:
assert submission.shape[0] == 300000
assert list(submission.columns) == ["id", "diagnosed_diabetes"]
assert submission["diagnosed_diabetes"].between(0, 1).all()
assert submission["id"].is_unique
